# Pipeline node 03 — Register in Model Registry and deploy with KServe

Production-parallel of notebook [`../05-register-and-deploy.ipynb`](../05-register-and-deploy.ipynb). Reads the model URI produced by node 02, registers a `ModelVersion`, creates/patches a KServe `InferenceService`, waits `Ready=True`.

**S3 contract**
- *Input:* `s3://$BUCKET/$MODEL_PREFIX/$MODEL_VERSION/` (object key is discovered or passed as `MODEL_URI`).
- *Output:* Registry entry + running InferenceService. The endpoint URL is written to stdout and to `endpoint.txt`.

**Env vars expected**
- `AWS_S3_BUCKET`, `MODEL_PREFIX`, `MODEL_VERSION` (must match node 02)
- `MODEL_REGISTRY_URL`, `MODEL_REGISTRY_TOKEN`, `MODEL_REGISTRY_PORT` (default `443`)
- `KUBERNETES_NAMESPACE`, `SERVING_RUNTIME` (default `ultralytics-yolo-runtime`), `SERVICE_ACCOUNT` (default `aws-connection-storage`)
- `MODEL_NAME` (default `ppe-yolov8n-vlm`), `JUPYTER_USER` (optional author tag)

In [ ]:
%%capture
!pip install model-registry kubernetes

In [ ]:
import os
import time
from pathlib import Path

BUCKET        = os.environ["AWS_S3_BUCKET"]
MODEL_PREFIX  = os.environ.get("MODEL_PREFIX",  "models/ppe-yolov8n-vlm").rstrip("/")
MODEL_VERSION = os.environ.get("MODEL_VERSION", "v1")
MODEL_NAME    = os.environ.get("MODEL_NAME",    "ppe-yolov8n-vlm")

MODEL_URI = os.environ.get("MODEL_URI") or f"s3://{BUCKET}/{MODEL_PREFIX}/{MODEL_VERSION}"
print("Model URI:", MODEL_URI)

## Register in the RHOAI Model Registry

In [ ]:
from model_registry import ModelRegistry

registry = ModelRegistry(
    server_address=os.environ["MODEL_REGISTRY_URL"],
    port=int(os.environ.get("MODEL_REGISTRY_PORT", 443)),
    author=os.environ.get("JUPYTER_USER", "pipeline"),
    user_token=os.environ.get("MODEL_REGISTRY_TOKEN"),
    is_secure=True,
)

rm = registry.register_model(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    model_format_name="pytorch",
    model_format_version="1",
    model_source_kind="s3",
    model_source_class="ultralytics",
    model_source_group="yolov8",
    model_source_id=MODEL_NAME,
    model_source_name="best.pt",
    uri=MODEL_URI,
    description="YOLOv8n fine-tuned on Qwen2.5-VL zero-shot annotations (pipeline run).",
    metadata={
        "classes":   "person,helmet,vest,gloves",
        "annotator": "Qwen2.5-VL-7B-Instruct",
        "run_mode":  "pipeline",
    },
)
print("Registered:", rm)

## Create or patch the KServe InferenceService

In [ ]:
from kubernetes import client, config

NAMESPACE       = os.environ["KUBERNETES_NAMESPACE"]
SERVING_RUNTIME = os.environ.get("SERVING_RUNTIME", "ultralytics-yolo-runtime")
SERVICE_ACCOUNT = os.environ.get("SERVICE_ACCOUNT", "aws-connection-storage")

isvc_manifest = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": MODEL_NAME,
        "namespace": NAMESPACE,
        "labels": {"opendatahub.io/dashboard": "true"},
        "annotations": {
            "serving.knative.openshift.io/enablePassthrough": "true",
            "sidecar.istio.io/inject": "true",
            "sidecar.istio.io/rewriteAppHTTPProbers": "true",
        },
    },
    "spec": {
        "predictor": {
            "serviceAccountName": SERVICE_ACCOUNT,
            "model": {
                "runtime": SERVING_RUNTIME,
                "modelFormat": {"name": "pytorch"},
                "storageUri": MODEL_URI,
                "resources": {
                    "requests": {"cpu": "500m", "memory": "1Gi"},
                    "limits":   {"cpu": "2",   "memory": "4Gi"},
                },
            },
        }
    },
}

try:
    config.load_incluster_config()
except config.ConfigException:
    config.load_kube_config()

api = client.CustomObjectsApi()
group, version, plural = "serving.kserve.io", "v1beta1", "inferenceservices"
try:
    api.create_namespaced_custom_object(group, version, NAMESPACE, plural, isvc_manifest)
    print("Created InferenceService", MODEL_NAME)
except client.ApiException as e:
    if e.status != 409:
        raise
    api.patch_namespaced_custom_object(group, version, NAMESPACE, plural, MODEL_NAME, isvc_manifest)
    print("Patched existing InferenceService", MODEL_NAME)

In [ ]:
TIMEOUT_S = 600
deadline = time.time() + TIMEOUT_S
url = None
while time.time() < deadline:
    isvc = api.get_namespaced_custom_object(group, version, NAMESPACE, plural, MODEL_NAME)
    status = isvc.get("status", {})
    ready = next((c for c in status.get("conditions", []) if c.get("type") == "Ready"), None)
    if ready and ready.get("status") == "True":
        url = status.get("url") or status.get("address", {}).get("url")
        break
    print(f"  … waiting, Ready={ready.get('status') if ready else 'unknown'}")
    time.sleep(10)

if not url:
    raise TimeoutError(f"InferenceService {MODEL_NAME} did not reach Ready=True in {TIMEOUT_S}s")

print("Endpoint:", url)
Path("endpoint.txt").write_text(url)